# Supernova Colheita
### Assistente Virtual com Inteligência Artificial e RAG para Agricultura Familiar

Projeto desenvolvido para a Global Solution FIAP 2026/2.

O sistema utiliza Google Gemini e busca por similaridade TF-IDF para auxiliar pequenos produtores rurais com informações sobre o Programa Nacional de Alimentação Escolar (PNAE).

## 1. Instalação das Bibliotecas

In [23]:

!pip -q install -U google-genai pypdf scikit-learn


## 	2. Configuração da API do Google Gemini


In [24]:

from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY_3")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Cliente Gemini configurado com sucesso!")


Cliente Gemini configurado com sucesso!


## 3. Construção da Base de Conhecimento do PNAE

In [25]:


base_conhecimento = [
    {
        "titulo": "Programa Nacional de Alimentação Escolar",
        "conteudo": """
        O PNAE é um programa do FNDE que oferece alimentação
        escolar aos estudantes da educação básica pública.
        O programa incentiva a compra de alimentos produzidos
        pela agricultura familiar.
        """
    },
    {
        "titulo": "Agricultura Familiar e PNAE",
        "conteudo": """
        A partir de 2026, no mínimo 45% dos recursos federais
        repassados pelo FNDE para a alimentação escolar devem
        ser utilizados na aquisição de alimentos da agricultura
        familiar, conforme a legislação vigente.
        """
    },
    {
        "titulo": "Documentação para Participação",
        "conteudo": """
        Agricultores familiares interessados em fornecer alimentos
        para escolas públicas devem observar as exigências da
        chamada pública, incluindo documentação de identificação,
        comprovação de enquadramento na agricultura familiar,
        projeto de venda e requisitos sanitários aplicáveis.
        As exigências específicas dependem do edital e da
        modalidade de participação.
        """
    }
]

print("Base de conhecimento criada com sucesso!")
print("Total de documentos:", len(base_conhecimento))


Base de conhecimento criada com sucesso!
Total de documentos: 3


## 4. Preparação dos Documentos e Vetorização TF-IDF

In [26]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Preparar os documentos
documentos = [item["conteudo"] for item in base_conhecimento]

# Transformar os textos em vetores
vetorizador = TfidfVectorizer(
    stop_words=None,
    strip_accents="unicode"
)

matriz_tfidf = vetorizador.fit_transform(documentos)

# Função de busca
def buscar_documentos(pergunta, quantidade=2):
    pergunta_vetor = vetorizador.transform([pergunta])
    similaridades = cosine_similarity(
        pergunta_vetor, matriz_tfidf
    )[0]

    indices = similaridades.argsort()[::-1][:quantidade]

    return [
        {
            "titulo": base_conhecimento[i]["titulo"],
            "conteudo": base_conhecimento[i]["conteudo"],
            "similaridade": round(float(similaridades[i]), 4)
        }
        for i in indices
        if similaridades[i] > 0
    ]

# Teste
pergunta = "Quais documentos preciso para vender alimentos às escolas?"

resultados = buscar_documentos(pergunta)

for resultado in resultados:
    print("Documento:", resultado["titulo"])
    print("Similaridade:", resultado["similaridade"])
    print("---------------------------------")


Documento: Documentação para Participação
Similaridade: 0.3504
---------------------------------
Documento: Agricultura Familiar e PNAE
Similaridade: 0.1149
---------------------------------


## 5. Implementação da Busca por Similaridade

In [27]:

def buscar_documentos(pergunta, quantidade=2, limite=0.20):
    pergunta_vetor = vetorizador.transform([pergunta])

    similaridades = cosine_similarity(
        pergunta_vetor, matriz_tfidf
    )[0]

    indices = similaridades.argsort()[::-1][:quantidade]

    return [
        {
            "titulo": base_conhecimento[i]["titulo"],
            "conteudo": base_conhecimento[i]["conteudo"],
            "similaridade": round(float(similaridades[i]), 4)
        }
        for i in indices
        if similaridades[i] >= limite
    ]


# Testar duas perguntas
perguntas = [
    "Quais documentos preciso para vender alimentos às escolas?",
    "Quem ganhou a Copa do Mundo de 2002?"
]

for pergunta in perguntas:
    print("\nPERGUNTA:", pergunta)
    print("RESPOSTA:", perguntar_supernova(pergunta))
    print("---------------------------------")



PERGUNTA: Quais documentos preciso para vender alimentos às escolas?
RESPOSTA: Olá! Como assistente virtual do projeto Supernova Colheita, orientarei você com base nas informações disponíveis.

Para vender alimentos às escolas públicas pelo PNAE, você precisará observar as exigências da chamada pública, que incluem:

* Documentação de identificação;
* Comprovação de enquadramento na agricultura familiar;
* Projeto de venda;
* Requisitos sanitários aplicáveis.

Como as exigências específicas dependem do edital e da modalidade de participação, orientamos que você consulte o edital da chamada pública e as fontes oficiais para obter a lista exata dos documentos necessários.

Fontes consultadas: Documentação para Participação
Modelo utilizado: gemini-3.5-flash-lite
---------------------------------

PERGUNTA: Quem ganhou a Copa do Mundo de 2002?
Modelo gemini-3.5-flash-lite indisponível: 503
RESPOSTA: Olá! Como assistente virtual do projeto Supernova Colheita, meu objetivo é orientar sobre

## 6. Otimização da Busca e Tratamento de Palavras Comuns

In [28]:

# Remover palavras comuns que prejudicam a busca
stopwords_pt = [
    "a", "o", "as", "os", "de", "do", "da", "dos", "das",
    "em", "no", "na", "nos", "nas", "um", "uma",
    "e", "ou", "para", "por", "com", "que", "qual",
    "quais", "quem", "como", "é", "foi", "ser",
    "ao", "aos", "se", "preciso"
]

# Recriar o vetorizador
vetorizador = TfidfVectorizer(
    stop_words=stopwords_pt,
    strip_accents="unicode"
)

matriz_tfidf = vetorizador.fit_transform(documentos)

# Testar a busca corrigida
perguntas = [
    "Quais documentos preciso para vender alimentos às escolas?",
    "Quem ganhou a Copa do Mundo de 2002?"
]

for pergunta in perguntas:
    print("\nPERGUNTA:", pergunta)

    resultados = buscar_documentos(pergunta)

    if resultados:
        for item in resultados:
            print("Documento:", item["titulo"])
            print("Similaridade:", item["similaridade"])
    else:
        print("Nenhum documento relevante encontrado.")

    print("---------------------------------")



PERGUNTA: Quais documentos preciso para vender alimentos às escolas?
Documento: Documentação para Participação
Similaridade: 0.2149
---------------------------------

PERGUNTA: Quem ganhou a Copa do Mundo de 2002?
Nenhum documento relevante encontrado.
---------------------------------


## 7. Integração do RAG com a Inteligência Artificial

In [29]:

from google.genai import errors

MODELOS_IA = [
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",
    "gemini-3.5-flash"
]

def perguntar_supernova(pergunta):
    resultados = buscar_documentos(pergunta, quantidade=2)

    if not resultados:
        return "Não encontrei informações sobre isso na base de conhecimento."

    contexto = "\n\n".join(
        f"Fonte: {item['titulo']}\n{item['conteudo']}"
        for item in resultados
    )

    prompt = f"""
    Você é o assistente virtual do projeto Supernova Colheita.

    Oriente agricultores familiares sobre o PNAE.
    Responda em português brasileiro.

    Utilize somente as informações do CONTEXTO.
    Não invente documentos, leis ou exigências.
    Caso faltem informações, oriente a consultar
    o edital da chamada pública e as fontes oficiais.

    CONTEXTO:
    {contexto}

    PERGUNTA:
    {pergunta}
    """

    for modelo in MODELOS_IA:
        try:
            resposta = client.models.generate_content(
                model=modelo,
                contents=prompt
            )

            if not resposta.text:
                continue

            fontes = ", ".join(
                item["titulo"] for item in resultados
            )

            return (
                f"{resposta.text}\n\n"
                f"Fontes consultadas: {fontes}\n"
                f"Modelo utilizado: {modelo}"
            )

        except (errors.ClientError, errors.ServerError) as erro:
            print(f"Modelo {modelo} indisponível: {erro.code}")
            continue

    return "Os modelos de IA estão indisponíveis no momento."


# TESTE
print("SUPERNOVA COLHEITA - TESTE FINAL")
print("---------------------------------")
print(perguntar_supernova(
    "Quais documentos preciso para vender alimentos às escolas?"
))


SUPERNOVA COLHEITA - TESTE FINAL
---------------------------------
Olá! Como assistente virtual do projeto Supernova Colheita, orientarei você com base nas informações disponíveis.

Para vender alimentos às escolas públicas pelo PNAE, você precisará observar as exigências da chamada pública, que incluem:

* Documentação de identificação;
* Comprovação de enquadramento na agricultura familiar;
* Projeto de venda;
* Requisitos sanitários aplicáveis.

Como as exigências específicas dependem do edital e da modalidade de participação, orientamos que você consulte o edital da chamada pública e as fontes oficiais para obter a lista exata dos documentos necessários.

Fontes consultadas: Documentação para Participação
Modelo utilizado: gemini-3.5-flash-lite


## 8. Registro das Fontes de Referência

In [30]:

fontes_conhecimento = [
    {
        "instituicao": "FNDE",
        "fonte": "Programa Nacional de Alimentação Escolar (PNAE)",
        "url": "https://www.gov.br/fnde/pt-br/acesso-a-informacao/acoes-e-programas/programas/pnae",
        "utilizacao": "Informações gerais sobre o PNAE"
    },
    {
        "instituicao": "FNDE",
        "fonte": "Agricultura Familiar no PNAE",
        "url": "https://www.gov.br/fnde/pt-br",
        "utilizacao": "Orientações sobre aquisição de alimentos da agricultura familiar"
    }
]

print("FONTES DE REFERÊNCIA - SUPERNOVA COLHEITA")
print("---------------------------------")

for fonte in fontes_conhecimento:
    print("Instituição:", fonte["instituicao"])
    print("Fonte:", fonte["fonte"])
    print("URL:", fonte["url"])
    print("Utilização:", fonte["utilizacao"])
    print("---------------------------------")


FONTES DE REFERÊNCIA - SUPERNOVA COLHEITA
---------------------------------
Instituição: FNDE
Fonte: Programa Nacional de Alimentação Escolar (PNAE)
URL: https://www.gov.br/fnde/pt-br/acesso-a-informacao/acoes-e-programas/programas/pnae
Utilização: Informações gerais sobre o PNAE
---------------------------------
Instituição: FNDE
Fonte: Agricultura Familiar no PNAE
URL: https://www.gov.br/fnde/pt-br
Utilização: Orientações sobre aquisição de alimentos da agricultura familiar
---------------------------------


## 9. Execução dos Testes Automatizados

In [31]:

import json
from datetime import datetime

casos_teste = [
    {
        "id": 1,
        "pergunta": "Quais documentos preciso para vender alimentos às escolas?",
        "esperado": "Resposta sobre documentação do PNAE"
    },
    {
        "id": 2,
        "pergunta": "Qual é o percentual mínimo de compras da agricultura familiar no PNAE em 2026?",
        "esperado": "Informar 45%"
    },
    {
        "id": 3,
        "pergunta": "Quem ganhou a Copa do Mundo de 2002?",
        "esperado": "Rejeitar pergunta fora do assunto"
    }
]

relatorio = {
    "projeto": "Supernova Colheita",
    "data": datetime.now().isoformat(),
    "modelo_principal": MODELOS_IA[0],
    "testes": []
}

for caso in casos_teste:
    print(f"Executando teste {caso['id']}...")

    resposta = perguntar_supernova(caso["pergunta"])

    relatorio["testes"].append({
        **caso,
        "resposta": resposta
    })

with open("relatorio_testes.json", "w", encoding="utf-8") as arquivo:
    json.dump(relatorio, arquivo, ensure_ascii=False, indent=4)

print("\nRelatório de testes salvo com sucesso!")
print("Arquivo: relatorio_testes.json")


Executando teste 1...
Executando teste 2...
Executando teste 3...

Relatório de testes salvo com sucesso!
Arquivo: relatorio_testes.json


## 10. Avaliação dos Resultados e Métricas

In [32]:

import json

with open("relatorio_testes.json", "r", encoding="utf-8") as arquivo:
    relatorio = json.load(arquivo)

aprovados = 0

for teste in relatorio["testes"]:
    resposta = teste["resposta"].lower()

    if teste["id"] == 1:
        passou = (
            "document" in resposta
            and "projeto de venda" in resposta
        )

    elif teste["id"] == 2:
        passou = "45%" in resposta

    elif teste["id"] == 3:
        passou = (
            "não encontrei informações" in resposta
        )

    else:
        passou = False

    teste["status"] = "APROVADO" if passou else "REVISAR"

    if passou:
        aprovados += 1

    print(f"Teste {teste['id']}: {teste['status']}")

total = len(relatorio["testes"])

print("---------------------------------")
print(f"Resultado: {aprovados}/{total} testes aprovados")

with open("relatorio_testes.json", "w", encoding="utf-8") as arquivo:
    json.dump(relatorio, arquivo, ensure_ascii=False, indent=4)


Teste 1: APROVADO
Teste 2: APROVADO
Teste 3: APROVADO
---------------------------------
Resultado: 3/3 testes aprovados


## 11. Download do Relatório de Testes

In [33]:
from google.colab import files

files.download("relatorio_testes.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>